In [6]:
# ============================================================
# RAG CONFIGURATION REGENERATION
# ============================================================
#
# PURPOSE
# -------
# Regenerate the SAME 9 RAG configuration summaries using
# one final, explicitly documented summary-generation model.
#
# This notebook DOES NOT:
# - change the selected patient
# - change preprocessing
# - change chunking definitions
# - change embedding models
# - change the retrieval query
# - change Top-K
# - change chronological reordering
#
# It only regenerates the 9 summary outputs under one final
# OpenAI generation model so downstream evaluations are
# internally consistent and reproducible.
#
# FINAL SUMMARY GENERATOR
# -----------------------
# Provider: OpenAI
# Model: gpt-5.6-luna
# Temperature: model default
#
# OUTPUT
# ------
# rag_configuration_summaries_final.json
# ============================================================

In [7]:

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from google import genai
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer

In [9]:
# ============================================================
# FROZEN EXPERIMENT SETTINGS
# ============================================================

SUMMARY_PROVIDER = "OpenAI"
SUMMARY_MODEL = "gpt-5.6-luna"

GEMINI_EMBEDDING_MODEL = "gemini-embedding-001"
BGE_EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

TOP_K = 20

RAG_QUERY = """
Retrieve the clinically relevant information needed to produce a
comprehensive longitudinal summary of this patient's clinical history,
including major diagnoses, treatments, investigations, clinical
progression, and outcomes.
""".strip()

CHUNKING_STRATEGIES = [
    "Whole",
    "Fixed",
    "Section",
]

EMBEDDING_STRATEGIES = [
    "Gemini",
    "BGE",
    "MedCPT",
]

print("==========================================")
print("RAG CONFIGURATION REGENERATION")
print("==========================================")
print("Summary provider :", SUMMARY_PROVIDER)
print("Summary model    :", SUMMARY_MODEL)
print("Top-K            :", TOP_K)
print("Chunking         :", CHUNKING_STRATEGIES)
print("Embeddings       :", EMBEDDING_STRATEGIES)
print("==========================================")

RAG CONFIGURATION REGENERATION
Summary provider : OpenAI
Summary model    : gpt-5.6-luna
Top-K            : 20
Chunking         : ['Whole', 'Fixed', 'Section']
Embeddings       : ['Gemini', 'BGE', 'MedCPT']


In [10]:
# ============================================================
# FINAL SUMMARY-GENERATION CLIENT
#
# This client goes DIRECTLY to OpenAI.
# It does NOT use Groq.
# ============================================================

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

print("Client class :", type(openai_client).__name__)
print("Client module:", type(openai_client).__module__)
print("Provider     :", SUMMARY_PROVIDER)
print("Model        :", SUMMARY_MODEL)

Client class : OpenAI
Client module: openai
Provider     : OpenAI
Model        : gpt-5.6-luna


In [11]:
# ============================================================
# SMALL CONNECTION TEST
#
# This confirms:
# 1. OPENAI_API_KEY works.
# 2. The frozen model is accessible.
# 3. We are not accidentally calling Groq.
# ============================================================

test_response = openai_client.chat.completions.create(
    model=SUMMARY_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Return exactly: OpenAI summary model ready"
        }
    ]
)

print(test_response.choices[0].message.content)

OpenAI summary model ready


In [13]:
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

In [14]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))

In [15]:
# ============================================================
# LOAD + PREPROCESS SOURCE NOTES
#
# Same preprocessing used in the original RAG experiment.
# ============================================================

notes = pd.read_csv(DATA_DIR / "raw" / "clinical_notes.csv")


notes_clean = notes[
    notes["clean_note_text"].astype(str).str.strip() != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(["person_id", "creation_timestamp"])
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first"
    )
    .reset_index(drop=True)
)

# SELECTED_PERSON_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f" # Long Pilot Case 
SELECTED_PERSON_ID = "c87e610e-ac2a-48cf-ac32-054f3e595498" # Short Pilot Case
# SELECTED_PERSON_ID = "ce0046dc-0ad3-4710-8147-549793c58b44" # Medium Pilot Case


patient_notes = (
    notes_dedup[
        notes_dedup["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Original notes:", len(notes))
print("After cleaning:", len(notes_clean))
print("After deduplication:", len(notes_dedup))
print("Selected patient notes:", len(patient_notes))

Original notes: 1602
After cleaning: 1595
After deduplication: 1103
Selected patient notes: 10


In [32]:
# ============================================================
# BUILD THE THREE FROZEN CHUNKING STRATEGIES
# ============================================================
#
# 1. WHOLE NOTE
#    One clinical note = one chunk
#
# 2. FIXED-SIZE
#    150 words with 30-word overlap
#
# 3. SECTION-AWARE
#    Split notes by section structure / headings
#
# These definitions should match the original 3x3 RAG experiment.
# ============================================================

In [16]:
# -------------------------
# 1. WHOLE-NOTE CHUNKS
# -------------------------

whole_chunks = (
    patient_notes[
        [
            "person_id",
            "creation_timestamp",
            "clean_note_text"
        ]
    ]
    .copy()
    .rename(columns={
        "clean_note_text": "chunk_text"
    })
)

whole_chunks["chunk_id"] = range(len(whole_chunks))

print("Whole-note chunks:", len(whole_chunks))

Whole-note chunks: 10


In [17]:
# -------------------------
# 2. FIXED-SIZE CHUNKS
# -------------------------

FIXED_CHUNK_SIZE = 150
FIXED_CHUNK_OVERLAP = 30


def chunk_text_fixed(text, chunk_size=150, overlap=30):
    words = str(text).split()

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        if chunk.strip():
            chunks.append(chunk)

        if end >= len(words):
            break

        start += chunk_size - overlap

    return chunks

In [18]:
fixed_records = []

for note_index, row in patient_notes.iterrows():

    note_chunks = chunk_text_fixed(
        row["clean_note_text"],
        chunk_size=FIXED_CHUNK_SIZE,
        overlap=FIXED_CHUNK_OVERLAP
    )

    for chunk_index, chunk_text in enumerate(note_chunks):

        fixed_records.append({
            "person_id": row["person_id"],
            "creation_timestamp": row["creation_timestamp"],
            "source_note_index": int(note_index),
            "within_note_chunk_index": chunk_index,
            "chunk_text": chunk_text
        })

fixed_chunks = pd.DataFrame(
    fixed_records
)

fixed_chunks["chunk_id"] = range(
    len(fixed_chunks)
)

print("Fixed-size chunks:", len(fixed_chunks))

Fixed-size chunks: 16


In [37]:
# ============================================================
# CHUNKING STRATEGY 3 — SECTION-AWARE
#
# Uses a fixed list of recognized clinical section headings.
#
# Rules:
# - Keep heading + section content together.
# - Remove heading-only empty sections.
# - Preserve any text before the first recognized heading.
# - If fewer than two usable section boundaries are detected,
#   keep the complete note as one chunk.
#
# IMPORTANT:
# This is the SAME section-aware chunking logic used in the
# original 3 x 3 RAG configuration experiment.
# ============================================================

In [19]:
SECTION_HEADINGS = [
    "Presenting Complaint",
    "History of Presenting Illness",
    "History of Present Illness",
    "HPI",
    "Review of Systems",
    "Past Medical History",
    "PMH",
    "Medications",
    "Medication",
    "Allergies",
    "Social History",
    "Family History",
    "On Examination",
    "Examination",
    "Observations",
    "Investigations",
    "Test Results",
    "Results",
    "Assessment",
    "Impression",
    "Diagnosis",
    "Treatment",
    "Plan",
]

SECTION_PATTERN = re.compile(
    rf"(?im)^(?:{'|'.join(map(re.escape, SECTION_HEADINGS))})\s*:?\s*$"
)

In [20]:
def is_heading_only(text: str) -> bool:
    """
    Return True when a chunk contains only a recognized
    section heading and no clinical content.
    """
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if len(lines) != 1:
        return False

    return bool(
        SECTION_PATTERN.fullmatch(lines[0])
    )

In [21]:
def split_by_sections(text: str) -> list[str]:
    """
    Split one clinical note using the frozen section headings.

    If usable section boundaries are not detected,
    return the complete note unchanged.
    """
    matches = list(
        SECTION_PATTERN.finditer(text)
    )

    if len(matches) < 2:
        return [text]

    chunks = []

    prefix = text[:matches[0].start()].strip()

    if prefix:
        chunks.append(prefix)

    for index, match in enumerate(matches):

        start = match.start()

        if index + 1 < len(matches):
            end = matches[index + 1].start()
        else:
            end = len(text)

        section = text[start:end].strip()

        if section and not is_heading_only(section):
            chunks.append(section)

    return chunks

In [22]:
def create_section_chunks(
    notes_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Create section-aware chunks from clinical notes.

    Notes without detectable section structure remain whole.
    """
    records = []

    for row in notes_df.itertuples(index=False):

        text_chunks = split_by_sections(
            row.clean_note_text
        )

        for chunk_index, chunk_text in enumerate(text_chunks):

            records.append(
                {
                    "person_id": row.person_id,
                    "admission_id": row.admission_id,
                    "clinical_note_id": row.clinical_note_id,
                    "creation_timestamp": row.creation_timestamp,
                    "note_subject": row.note_subject,
                    "note_type": row.note_type,
                    "chunk_index": chunk_index,
                    "chunk_id": (
                        f"{row.clinical_note_id}"
                        f"_section_{chunk_index}"
                    ),
                    "chunk_strategy": "section",
                    "chunk_text": chunk_text,
                }
            )

    return pd.DataFrame(records)

In [23]:
section_chunks = create_section_chunks(
    patient_notes
)

print("Section-aware chunks:", len(section_chunks))

Section-aware chunks: 44


In [24]:
chunk_summary = pd.DataFrame(
    {
        "strategy": [
            "Whole note",
            "Fixed words",
            "Section aware",
        ],
        "num_chunks": [
            len(whole_chunks),
            len(fixed_chunks),
            len(section_chunks),
        ],
        "avg_words_per_chunk": [
            whole_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),

            fixed_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),

            section_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),
        ],
    }
)

display(chunk_summary)

,strategy,num_chunks,avg_words_per_chunk
0,Whole note,10,138.000000
1,Fixed words,16,97.500000
2,Section aware,44,31.295455


In [47]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [48]:
# ============================================================
# EMBEDDING MODEL SETUP
# ============================================================
#
# These are the SAME three embedding approaches used in the
# original 3 x 3 RAG configuration experiment.
#
# Gemini:
#   gemini-embedding-001
#
# BGE:
#   BAAI/bge-base-en-v1.5
#
# MedCPT:
#   Separate Query Encoder and Article Encoder
#
# IMPORTANT:
# - Document/chunk embeddings use the Article Encoder for MedCPT.
# - Retrieval queries use the Query Encoder for MedCPT.
# ============================================================

In [25]:

import torch

# -------------------------
# Gemini
# -------------------------

gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

GEMINI_EMBEDDING_MODEL = "gemini-embedding-001"


# -------------------------
# BGE
# -------------------------

BGE_MODEL_PATH = PROJECT_ROOT / "models" / "bge-base-en-v1.5"

bge_model = SentenceTransformer(
    str(BGE_MODEL_PATH)
)


# -------------------------
# MedCPT
# -------------------------

MEDCPT_QUERY_PATH = (
    PROJECT_ROOT
    / "models"
    / "MedCPT-Query-Encoder"
)

MEDCPT_ARTICLE_PATH = (
    PROJECT_ROOT
    / "models"
    / "MedCPT-Article-Encoder"
)

query_tokenizer = AutoTokenizer.from_pretrained(
    MEDCPT_QUERY_PATH,
    local_files_only=True
)

query_model = AutoModel.from_pretrained(
    MEDCPT_QUERY_PATH,
    local_files_only=True
)

article_tokenizer = AutoTokenizer.from_pretrained(
    MEDCPT_ARTICLE_PATH,
    local_files_only=True
)

article_model = AutoModel.from_pretrained(
    MEDCPT_ARTICLE_PATH,
    local_files_only=True
)

query_model.eval()
article_model.eval()

print("Gemini embedding client ready.")
print("BGE model loaded.")
print("MedCPT query/article models loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Gemini embedding client ready.
BGE model loaded.
MedCPT query/article models loaded.


In [50]:
# ============================================================
# DOCUMENT / CHUNK EMBEDDING FUNCTIONS
# ============================================================

In [26]:
import time 

def embed_gemini_texts(texts):
    """
    Generate Gemini embeddings one text at a time.

    Each clinical chunk is embedded independently using
    gemini-embedding-001.

    We intentionally avoid batchEmbedContents because the
    current Gemini API credentials do not permit that endpoint.
    """
    embeddings = []

    for index, text in enumerate(texts, start=1):
        print(f"Gemini embedding: {index}/{len(texts)}")

        response = gemini_client.models.embed_content(
            model=GEMINI_EMBEDDING_MODEL,
            contents=text,
        )

        embedding = response.embeddings[0].values
        embeddings.append(embedding)
        time.sleep(1.0)

    return np.array(embeddings)

In [27]:
def embed_bge_texts(texts):
    """
    Embed clinical chunks with BGE.

    Embeddings are L2-normalized so dot product can later be
    used as cosine similarity.
    """
    return bge_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

In [28]:
# -------------------------
# MedCPT
# -------------------------

def embed_medcpt_articles(texts):
    """
    Embed clinical chunks using the MedCPT Article Encoder.

    MedCPT is BERT-based and supports a maximum sequence length
    of 512 tokens, so inputs are truncated to 512 tokens.
    """
    inputs = article_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )

    with torch.no_grad():
        outputs = article_model(**inputs)

    embeddings = outputs.last_hidden_state[:, 0, :]

    return embeddings.cpu().numpy()

In [29]:
# -------------------------
# MedCPT Query Encoder
# -------------------------

def embed_medcpt_query(query_text):
    """
    Encode the retrieval query with the MedCPT Query Encoder.
    """
    inputs = query_tokenizer(
        query_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )

    with torch.no_grad():
        outputs = query_model(**inputs)

    embedding = outputs.last_hidden_state[:, 0, :]

    return embedding.cpu().numpy()[0]

In [55]:
# ============================================================
# GENERATE EMBEDDINGS FOR ALL 3 x 3 RAG CONFIGURATIONS
# ============================================================
#
# Chunking strategies:
#   1. Whole note
#   2. Fixed-size
#   3. Section-aware
#
# Embedding approaches:
#   1. Gemini
#   2. BGE
#   3. MedCPT
#
# This creates 9 document-embedding matrices.
#
# IMPORTANT:
# No summaries are generated in this cell.
# No LLM evaluation occurs in this cell.
# ============================================================


# ------------------------------------------------------------
# Extract chunk text once for each chunking strategy
# ------------------------------------------------------------

In [30]:

# ------------------------------------------------------------
#     Chunks 
# ------------------------------------------------------------

whole_texts = (
    whole_chunks["chunk_text"]
    .astype(str)
    .tolist()
)

fixed_texts = (
    fixed_chunks["chunk_text"]
    .astype(str)
    .tolist()
)

section_texts = (
    section_chunks["chunk_text"]
    .astype(str)
    .tolist()
)

In [31]:
# ============================================================
# GEMINI EMBEDDINGS — WHOLE NOTE
# ============================================================

print("Generating Whole + Gemini embeddings...")

whole_gemini_embeddings = embed_gemini_texts(
    whole_texts
)

print("Whole + Gemini complete.")
print("Shape:", whole_gemini_embeddings.shape)

Generating Whole + Gemini embeddings...
Gemini embedding: 1/10
Gemini embedding: 2/10
Gemini embedding: 3/10
Gemini embedding: 4/10
Gemini embedding: 5/10
Gemini embedding: 6/10
Gemini embedding: 7/10
Gemini embedding: 8/10
Gemini embedding: 9/10
Gemini embedding: 10/10
Whole + Gemini complete.
Shape: (10, 3072)


In [32]:
# ============================================================
# GEMINI EMBEDDINGS — FIXED SIZE
# ============================================================

print("Generating Fixed + Gemini embeddings...")

fixed_gemini_embeddings = embed_gemini_texts(
    fixed_texts
)

print("Fixed + Gemini complete.")
print("Shape:", fixed_gemini_embeddings.shape)

Generating Fixed + Gemini embeddings...
Gemini embedding: 1/16
Gemini embedding: 2/16
Gemini embedding: 3/16
Gemini embedding: 4/16
Gemini embedding: 5/16
Gemini embedding: 6/16
Gemini embedding: 7/16
Gemini embedding: 8/16
Gemini embedding: 9/16
Gemini embedding: 10/16
Gemini embedding: 11/16
Gemini embedding: 12/16
Gemini embedding: 13/16
Gemini embedding: 14/16
Gemini embedding: 15/16
Gemini embedding: 16/16
Fixed + Gemini complete.
Shape: (16, 3072)


In [33]:
# ============================================================
# GEMINI EMBEDDINGS — SECTION AWARE
# ============================================================

print("Generating Section + Gemini embeddings...")

section_gemini_embeddings = embed_gemini_texts(
    section_texts
)

print("Section + Gemini complete.")
print("Shape:", section_gemini_embeddings.shape)

Generating Section + Gemini embeddings...
Gemini embedding: 1/44
Gemini embedding: 2/44
Gemini embedding: 3/44
Gemini embedding: 4/44
Gemini embedding: 5/44
Gemini embedding: 6/44
Gemini embedding: 7/44
Gemini embedding: 8/44
Gemini embedding: 9/44
Gemini embedding: 10/44
Gemini embedding: 11/44
Gemini embedding: 12/44
Gemini embedding: 13/44
Gemini embedding: 14/44
Gemini embedding: 15/44
Gemini embedding: 16/44
Gemini embedding: 17/44
Gemini embedding: 18/44
Gemini embedding: 19/44
Gemini embedding: 20/44
Gemini embedding: 21/44
Gemini embedding: 22/44
Gemini embedding: 23/44
Gemini embedding: 24/44
Gemini embedding: 25/44
Gemini embedding: 26/44
Gemini embedding: 27/44
Gemini embedding: 28/44
Gemini embedding: 29/44
Gemini embedding: 30/44
Gemini embedding: 31/44
Gemini embedding: 32/44
Gemini embedding: 33/44
Gemini embedding: 34/44
Gemini embedding: 35/44
Gemini embedding: 36/44
Gemini embedding: 37/44
Gemini embedding: 38/44
Gemini embedding: 39/44
Gemini embedding: 40/44
Gemini 

In [34]:
# ------------------------------------------------------------
# BGE embeddings
# ------------------------------------------------------------

print("\nGenerating BGE embeddings...")

whole_bge_embeddings = embed_bge_texts(whole_texts)
fixed_bge_embeddings = embed_bge_texts(fixed_texts)
section_bge_embeddings = embed_bge_texts(section_texts)


print("BGE complete.")


Generating BGE embeddings...
BGE complete.


In [35]:
# ------------------------------------------------------------
# MedCPT embeddings
# ------------------------------------------------------------

print("\nGenerating MedCPT embeddings...")

whole_medcpt_embeddings = embed_medcpt_articles(whole_texts)
fixed_medcpt_embeddings = embed_medcpt_articles(fixed_texts)
section_medcpt_embeddings = embed_medcpt_articles(section_texts)

print("MedCPT complete.")


Generating MedCPT embeddings...
MedCPT complete.


In [36]:
# ============================================================
# FINAL EMBEDDING SANITY CHECK
# ============================================================

embedding_shapes = pd.DataFrame(
    [
        ["Whole + Gemini", len(whole_chunks), whole_gemini_embeddings.shape],
        ["Whole + BGE", len(whole_chunks), whole_bge_embeddings.shape],
        ["Whole + MedCPT", len(whole_chunks), whole_medcpt_embeddings.shape],

        ["Fixed + Gemini", len(fixed_chunks), fixed_gemini_embeddings.shape],
        ["Fixed + BGE", len(fixed_chunks), fixed_bge_embeddings.shape],
        ["Fixed + MedCPT", len(fixed_chunks), fixed_medcpt_embeddings.shape],

        ["Section + Gemini", len(section_chunks), section_gemini_embeddings.shape],
        ["Section + BGE", len(section_chunks), section_bge_embeddings.shape],
        ["Section + MedCPT", len(section_chunks), section_medcpt_embeddings.shape],
    ],
    columns=[
        "configuration",
        "num_chunks",
        "embedding_shape",
    ],
)

display(embedding_shapes)

,configuration,num_chunks,embedding_shape
0,Whole + Gemini,10,"(10, 3072)"
1,Whole + BGE,10,"(10, 768)"
2,Whole + MedCPT,10,"(10, 768)"
3,Fixed + Gemini,16,"(16, 3072)"
4,Fixed + BGE,16,"(16, 768)"
5,Fixed + MedCPT,16,"(16, 768)"
6,Section + Gemini,44,"(44, 3072)"
7,Section + BGE,44,"(44, 768)"
8,Section + MedCPT,44,"(44, 768)"


In [37]:
# Gemini query embedding
gemini_query_embedding = embed_gemini_texts(
    [RAG_QUERY]
)[0]

Gemini embedding: 1/1


In [38]:
# BGE query embedding
bge_query_embedding = embed_bge_texts(
    [RAG_QUERY]
)[0]


In [39]:
# MedCPT query embedding
medcpt_query_embedding = embed_medcpt_query(
    RAG_QUERY
)

In [40]:
def retrieve_top_k_chunks(
    chunks_df,
    chunk_embeddings,
    query_embedding,
    top_k=20,
):
    """
    Retrieve the top-k chunks by cosine similarity and then
    reorder only those retrieved chunks chronologically.
    """

    # Cosine similarity because vectors are already normalized
    # where applicable; sklearn handles all cases consistently.
    scores = cosine_similarity(
        query_embedding.reshape(1, -1),
        chunk_embeddings,
    )[0]

    k = min(top_k, len(chunks_df))

    top_indices = np.argsort(scores)[::-1][:k]

    retrieved = chunks_df.iloc[top_indices].copy()

    retrieved["similarity_score"] = scores[top_indices]

    # Convert to datetime before chronological sorting.
    retrieved["creation_timestamp"] = pd.to_datetime(
        retrieved["creation_timestamp"],
        dayfirst=True,
    )

    retrieved = (
        retrieved
        .sort_values("creation_timestamp")
        .reset_index(drop=True)
    )

    return retrieved

In [41]:
retrieval_results = {
    "Whole + Gemini": retrieve_top_k_chunks(
        whole_chunks,
        whole_gemini_embeddings,
        gemini_query_embedding,
        TOP_K,
    ),
    "Whole + BGE": retrieve_top_k_chunks(
        whole_chunks,
        whole_bge_embeddings,
        bge_query_embedding,
        TOP_K,
    ),
    "Whole + MedCPT": retrieve_top_k_chunks(
        whole_chunks,
        whole_medcpt_embeddings,
        medcpt_query_embedding,
        TOP_K,
    ),
    "Fixed + Gemini": retrieve_top_k_chunks(
        fixed_chunks,
        fixed_gemini_embeddings,
        gemini_query_embedding,
        TOP_K,
    ),
    "Fixed + BGE": retrieve_top_k_chunks(
        fixed_chunks,
        fixed_bge_embeddings,
        bge_query_embedding,
        TOP_K,
    ),
    "Fixed + MedCPT": retrieve_top_k_chunks(
        fixed_chunks,
        fixed_medcpt_embeddings,
        medcpt_query_embedding,
        TOP_K,
    ),
    "Section + Gemini": retrieve_top_k_chunks(
        section_chunks,
        section_gemini_embeddings,
        gemini_query_embedding,
        TOP_K,
    ),
    "Section + BGE": retrieve_top_k_chunks(
        section_chunks,
        section_bge_embeddings,
        bge_query_embedding,
        TOP_K,
    ),
    "Section + MedCPT": retrieve_top_k_chunks(
        section_chunks,
        section_medcpt_embeddings,
        medcpt_query_embedding,
        TOP_K,
    ),
}

In [42]:
for config_name, retrieved_df in retrieval_results.items():
    print(
        f"{config_name}: "
        f"{len(retrieved_df)} retrieved chunks"
    )

Whole + Gemini: 10 retrieved chunks
Whole + BGE: 10 retrieved chunks
Whole + MedCPT: 10 retrieved chunks
Fixed + Gemini: 16 retrieved chunks
Fixed + BGE: 16 retrieved chunks
Fixed + MedCPT: 16 retrieved chunks
Section + Gemini: 20 retrieved chunks
Section + BGE: 20 retrieved chunks
Section + MedCPT: 20 retrieved chunks


In [43]:
display(
    retrieval_results["Whole + BGE"][
        [
            "chunk_id",
            "creation_timestamp",
            "similarity_score",
        ]
    ]
)

,chunk_id,creation_timestamp,similarity_score
0,0,2026-01-04 10:10:00,0.504262
1,1,2026-01-04 10:30:00,0.490179
2,2,2026-01-04 11:00:00,0.519957
3,3,2026-01-04 11:30:00,0.532903
4,4,2026-01-04 12:00:00,0.476205
5,5,2026-01-04 13:30:00,0.481957
6,6,2026-01-04 15:00:00,0.541346
7,7,2026-01-04 16:30:00,0.480750
8,8,2026-01-05 08:00:00,0.518664
9,9,2026-01-05 09:30:00,0.527692


In [44]:
def build_rag_context(retrieved_df):
    context_parts = []

    for index, row in retrieved_df.iterrows():
        timestamp = row["creation_timestamp"].strftime(
            "%Y-%m-%d %H:%M"
        )

        context_parts.append(
            f"""[SOURCE CHUNK {index + 1}]
Creation timestamp: {timestamp}

{row["chunk_text"]}"""
        )

    return "\n\n---\n\n".join(context_parts)

In [45]:
rag_contexts = {
    config_name: build_rag_context(retrieved_df)
    for config_name, retrieved_df in retrieval_results.items()
}

for config_name, context in rag_contexts.items():
    print(
        f"{config_name}: "
        f"{len(context)} characters"
    )

Whole + Gemini: 9630 characters
Whole + BGE: 9630 characters
Whole + MedCPT: 9630 characters
Fixed + Gemini: 11060 characters
Fixed + BGE: 11060 characters
Fixed + MedCPT: 11060 characters
Section + Gemini: 5383 characters
Section + BGE: 5965 characters
Section + MedCPT: 3108 characters


In [46]:
# ============================================================
# RAG CONTEXT SANITY CHECK
# ============================================================
#
# Context length is expected to differ across chunking
# strategies because Top-K is fixed at 20 chunks rather than
# a fixed number of characters/tokens.
#
# Do NOT truncate/equalize contexts here. Differences in
# retrieved evidence volume are part of the configuration
# comparison.
# ============================================================

context_summary = pd.DataFrame(
    [
        {
            "configuration": config_name,
            "retrieved_chunks": len(retrieval_results[config_name]),
            "context_characters": len(context),
        }
        for config_name, context in rag_contexts.items()
    ]
)

display(context_summary)

,configuration,retrieved_chunks,context_characters
0,Whole + Gemini,10,9630
1,Whole + BGE,10,9630
2,Whole + MedCPT,10,9630
3,Fixed + Gemini,16,11060
4,Fixed + BGE,16,11060
5,Fixed + MedCPT,16,11060
6,Section + Gemini,20,5383
7,Section + BGE,20,5965
8,Section + MedCPT,20,3108


In [81]:
# ============================================================
# FINAL RAG CONFIGURATION SUMMARY GENERATION
# ============================================================
#
# PURPOSE
# -------
# Generate one longitudinal clinical summary for each of the
# 9 RAG configurations created above:
#
#   3 chunking strategies:
#       - Whole note
#       - Fixed-size
#       - Section-aware
#
#   x
#
#   3 embedding approaches:
#       - Gemini
#       - BGE
#       - MedCPT
#
# = 9 total RAG configurations.
#
#
# RETRIEVAL PIPELINE
# ------------------
# For every configuration:
#
# 1. The SAME frozen retrieval query is used.
#
# 2. The query is embedded using the embedding approach
#    corresponding to that configuration.
#
# 3. Cosine similarity is calculated between the query
#    embedding and all candidate chunk embeddings.
#
# 4. The Top-20 most similar chunks are retrieved.
#
# 5. The selected Top-20 chunks are reordered by their
#    original creation timestamp before generation.
#
#    IMPORTANT:
#    Similarity determines WHICH chunks are selected.
#    Chronology determines HOW the selected evidence is
#    presented to the summarization model.
#
# 6. Each retrieved chunk is supplied with its creation
#    timestamp and clinical text.
#
# 7. Similarity scores are NOT supplied to the summarization
#    model.
#
#
# WHY CONTEXT LENGTHS DIFFER
# --------------------------
# Top-K is fixed at 20 CHUNKS rather than a fixed character
# or token budget.
#
# Therefore:
# - Whole-note retrieval produces longer contexts.
# - Fixed-size retrieval produces intermediate contexts.
# - Section-aware retrieval produces shorter contexts.
#
# Context lengths are intentionally NOT equalized because
# chunk granularity is part of the configuration comparison.
#
#
# GENERATION CONTROL
# ------------------
# All 9 retrieved contexts will now be summarized using:
#
# - the SAME generation model
# - the SAME system prompt
# - the SAME generation function
#
# Therefore, the experimental variables across the 9
# configurations remain chunking strategy and embedding
# approach.
#
# These regenerated summaries replace the earlier summaries
# that were generated through the previous Groq/GPT-OSS
# generation setup.
#
# All downstream evaluations must therefore be rerun using
# THESE final summaries.
# ============================================================

In [47]:
import sys
from pathlib import Path

cwd = Path.cwd()
proj = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(proj))

print("Added to sys.path:", sys.path[0])

Added to sys.path: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/notebooks


In [48]:
from src.llm.llm import generate_rag_summary

In [49]:
# ============================================================
# SUMMARY GENERATION CONFIGURATION
# ============================================================
#
# All 9 RAG configurations are summarized using the same
# generation model and prompt.
#
# Generation is handled by generate_rag_summary() in src.llm.llm.
# The llm.py module is configured to use the OpenAI client and
# gpt-5.6-luna.
#
# Keeping generation fixed ensures that the only experimental
# differences across the 9 configurations are:
#   1. Chunking strategy
#   2. Embedding approach
# ============================================================

from src.llm.llm import LLM_MODEL, LLM_PROVIDER, generate_rag_summary

print("Summary generator:", generate_rag_summary.__module__)
print("Provider:", LLM_PROVIDER)
print("Model:", LLM_MODEL)

Summary generator: src.llm.llm
Provider: OpenAI
Model: gpt-5.6-luna


In [91]:
# ============================================================
# GENERATE AND CHECKPOINT FINAL RAG SUMMARIES
# ============================================================
#
# PURPOSE
# -------
# Generate one summary for each of the 9 frozen RAG contexts.
#
# COST / RATE-LIMIT CONTROLS
# --------------------------
# - Exactly one LLM call is made per configuration.
# - Completed summaries are saved immediately.
# - Existing summaries are skipped if this cell is rerun.
# - A short delay is added between calls.
# - No automatic retries are used here to avoid accidental
#   duplicate/costly generations.
#
# IMPORTANT
# ---------
# All configurations use:
# - the same OpenAI generation model
# - the same system prompt
# - the same Top-20 retrieval setting
#
# Only chunking strategy and embedding approach differ.
# ============================================================

In [53]:
# ============================================================
# OUTPUT PATH — SHORT PILOT
# ============================================================

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "rag"
    / "short"
    / "rag_configuration_summaries.json"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# Load previous progress for THIS pilot patient only.
if OUTPUT_PATH.exists():
    with OUTPUT_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        rag_summaries = json.load(file)
else:
    rag_summaries = {}

print("OUTPUT_PATH:", OUTPUT_PATH)
print("Existing summaries:", len(rag_summaries))

OUTPUT_PATH: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/rag/short/rag_configuration_summaries.json
Existing summaries: 0


In [54]:
from config.prompts import SYSTEM_PROMPT

for configuration, context in rag_contexts.items():

    # Never pay for the same configuration twice.
    if configuration in rag_summaries:
        print(f"Skipping completed: {configuration}")
        continue

    print(f"\nGenerating: {configuration}")
    print(f"Context characters: {len(context):,}")

    try:
        summary = generate_rag_summary(
            context=context,
            prompt=SYSTEM_PROMPT,
        )

        rag_summaries[configuration] = {
            "summary": summary,
            "context_characters": len(context),
            "retrieved_chunks": len(
                retrieval_results[configuration]
            ),
            "model": LLM_MODEL,
            "provider": LLM_PROVIDER,
        }

        # Checkpoint immediately after every successful call.
        with OUTPUT_PATH.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                rag_summaries,
                file,
                indent=2,
                ensure_ascii=False,
            )

        print(f"Saved: {configuration}")

    except Exception as error:
        print(
            f"FAILED: {configuration}\n"
            f"{type(error).__name__}: {error}"
        )

        # Stop rather than repeatedly calling the API.
        break

    # Gentle spacing between generation requests.
    time.sleep(2)


Generating: Whole + Gemini
Context characters: 9,630
Saved: Whole + Gemini

Generating: Whole + BGE
Context characters: 9,630
Saved: Whole + BGE

Generating: Whole + MedCPT
Context characters: 9,630
Saved: Whole + MedCPT

Generating: Fixed + Gemini
Context characters: 11,060
Saved: Fixed + Gemini

Generating: Fixed + BGE
Context characters: 11,060
Saved: Fixed + BGE

Generating: Fixed + MedCPT
Context characters: 11,060
Saved: Fixed + MedCPT

Generating: Section + Gemini
Context characters: 5,383
Saved: Section + Gemini

Generating: Section + BGE
Context characters: 5,965
Saved: Section + BGE

Generating: Section + MedCPT
Context characters: 3,108
Saved: Section + MedCPT
